# Performance
Evaluates the performance/alignment (proportional match) of LLMs' generated distractors with human annotated ones (from Eedi).

Produces Table 2 and its full breakdown (Table 17, #correct/#repetitions), the model-breadth table (Table 24, Appendix C.3), and the correct-answer-reveal ablation numbers quoted in Section 4.3 (Eedi 0.52→0.56, SciQ 0.14→0.18, +6.4%/+30.7% relative).

In [ ]:
import os
import json
from tqdm import tqdm
from dotenv import load_dotenv

import pandas as pd
from openai import OpenAI

from src.datasets import ADataset, get_or_create_dataset
from src.equality import MathSemanticEqualityChecker, ScienceSemanticEqualityChecker, EqualityChecker
from src.evaluation import number_correct, number_repetitions, proportional_match, partial_match, exact_match, get_mean_and_ci
from src.model_configurations import gpt_4_1_mini_det_config

load_dotenv()

In [ ]:
equality_model_config = gpt_4_1_mini_det_config
equality_client = OpenAI(base_url=equality_model_config["base_url"], api_key=os.environ.get(equality_model_config["api_key_var"], None))

In [ ]:
def plot_results(data_folder: str, dataset: ADataset, filter_unsolvable: bool = False, filter_cutoff: bool = True):
    # -> Table 2 / Table 17 (proportional_match, number_correct, repetitions per setting)
    results_by_setting = {}

    results_folder = os.path.join(data_folder, "joint_results")
    for filename in sorted(os.listdir(results_folder)):
        if not filename.endswith("_responses_by_datapointid.json"): continue
        setting = filename.replace("_responses_by_datapointid.json", "")
        try: 
            results_by_setting[setting] = pd.read_csv(os.path.join(results_folder, f"{setting}_results.csv"))
        except Exception as e:
            print(f"Failed to load results for {setting}")

    if filter_unsolvable or filter_cutoff:
        results_by_setting = {
            setting: df[df["Id"].apply(lambda x: dataset[int(x)]["Problem"]["Solvable"])]
            for setting,df in results_by_setting.items()
        }

    for setting,result_df in results_by_setting.items():
        print(f"[{setting}]")

        for c in result_df.columns:
            if c == "Id" or c == "distractors": continue
            col = pd.to_numeric(result_df[c], errors="coerce").dropna()
            if len(col) == 0:
                continue
            mean,confidence = get_mean_and_ci(col)
            print(f"\t{c}: {mean:.3f} ± {confidence:.3f}")

## Eedi

In [ ]:
math_equivalence_check_path = "cache/math_semantic_equivalence_checker.pkl"
if os.path.exists(math_equivalence_check_path):
    print("Loading existing semantic equality checker")
    semantic_equality_checker = MathSemanticEqualityChecker.load(equality_client, math_equivalence_check_path)
else:
    semantic_equality_checker = MathSemanticEqualityChecker(equality_client, equality_model_config)
    semantic_equality_checker.save(math_equivalence_check_path)

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
print(f"We have {len(eedi_dataset)} EEDI questions")

datasets_by_datafolder = {
    "eedi_data": eedi_dataset
}

### Evaluation

In [ ]:
_dataset_lookup_cache = {}

def _dataset_lookups(dataset: ADataset):
    # Rebuilding these four dicts from `dataset` is O(len(dataset)) and was previously done
    # once per *file* (~30 result files) instead of once per *dataset* -- cache by identity.
    key = id(dataset)
    if key not in _dataset_lookup_cache:
        _dataset_lookup_cache[key] = (
            {str(dpid): dataset[dpid]["Problem"]["Question"] for dpid in range(len(dataset))},
            {str(dpid): dataset[dpid]["Choices"]["Distractors"] for dpid in range(len(dataset))},
            {str(dpid): dataset[dpid]["Choices"]["CorrectAnswer"] for dpid in range(len(dataset))},
            {str(dpid): dataset[dpid]["Problem"]["NumReasoningSteps"] for dpid in range(len(dataset))},
        )
    return _dataset_lookup_cache[key]


def analyze_responses(dataset: ADataset, responses_by_datapointid: dict, results_by_datapointid: dict, equality_checker: EqualityChecker):
    (problem_by_datapointid, gt_distractors_by_datapointid,
     gt_correct_by_datapointid, num_cor_sol_steps_by_datapointid) = _dataset_lookups(dataset)

    def compute_field(field, dpid, responses_of_datapointid, problem, responses, groundtruths, correct_answer):
        if field == "proportional_match":
            return proportional_match(equality_checker, problem, responses, groundtruths)
        if field == "exact_match":
            return exact_match(equality_checker, problem, responses, groundtruths)
        if field == "partial_match":
            return partial_match(equality_checker, problem, responses, groundtruths)
        if field == "distractors":
            return responses
        if field == "num_distractors":
            return len(responses)
        if field == "repetitions":
            return responses_of_datapointid.get("statistics", {}).get("repetitions", number_repetitions(equality_checker, problem, responses))
        if field == "number_correct":
            return responses_of_datapointid.get("statistics", {}).get("number_correct", number_correct(equality_checker, problem, responses, correct_answer))
        if field == "attempts":
            return responses_of_datapointid.get("statistics", {}).get("attempts", 1)
        if field == "problem_len_chars":
            return len(problem)
        if field == "num_cor_sol_steps_by_datapointid":
            return num_cor_sol_steps_by_datapointid[dpid]
        if field == "reasoning_len_chars":
            text = responses_of_datapointid.get("raw_reasoning") or responses_of_datapointid.get("step_by_step") or ""
            return len(text)
        if field == "answer_len_chars":
            return (sum(len(str(r)) for r in responses) / len(responses)) if responses else 0.0
        raise ValueError(f"Unknown field: {field}")

    fields = [
        "proportional_match", "exact_match", "partial_match", "distractors",
        "num_distractors", "repetitions", "number_correct", "attempts",
        "problem_len_chars", "num_cor_sol_steps_by_datapointid",
        "reasoning_len_chars", "answer_len_chars",
    ]

    # num_cor_sol_steps_by_datapointid is legitimately NaN on SciQ (no step-count difficulty
    # measure there) -- treating a NaN as "missing" for that field alone would force a full
    # equivalence-checker recompute of every field, on every row, on every re-run.
    ALWAYS_NULLABLE_FIELDS = {"num_cor_sol_steps_by_datapointid"}

    def is_missing(row, field):
        if field not in row:
            return True
        if field in ALWAYS_NULLABLE_FIELDS:
            return False
        v = row[field]
        try:
            if pd.isna(v):
                return True
        except (TypeError, ValueError):
            pass
        return False

    for dpid in tqdm(responses_by_datapointid.keys()):
        dpid = str(dpid)
        existing = dict(results_by_datapointid.get(dpid, {}))
        missing = [f for f in fields if is_missing(existing, f)]
        if not missing:
            results_by_datapointid[dpid] = existing
            continue

        responses_of_datapointid = responses_by_datapointid[dpid]
        problem = problem_by_datapointid[dpid]
        responses = [v for k, v in responses_of_datapointid.items() if k.endswith("_answer")]
        groundtruths = gt_distractors_by_datapointid[dpid]
        correct_answer = gt_correct_by_datapointid[dpid]

        for f in missing:
            existing[f] = compute_field(f, dpid, responses_of_datapointid, problem, responses, groundtruths, correct_answer)
        results_by_datapointid[dpid] = existing

    return pd.DataFrame([{"Id": dpid, **result} for dpid, result in results_by_datapointid.items()])    

In [ ]:
for data_folder,dataset in datasets_by_datafolder.items():
    results_folder = os.path.join(data_folder, "joint_results")
    
    for filename in os.listdir(results_folder):
        if not filename.endswith("_responses_by_datapointid.json"): continue
        
        print(f"Processing responses: {os.path.join(results_folder, filename)}")
        with open(os.path.join(results_folder, filename), "r+") as f:
            responses_by_datapointid = json.loads(f.read())

            results_file = os.path.join(results_folder, f"{filename.replace('_responses_by_datapointid.json', '')}_results.csv")
            
            # load existing results
            results_by_datapointid = {}
            if os.path.exists(results_file):
                results_df = pd.read_csv(results_file)
                results_by_datapointid = {
                    str(row["Id"]): {k: row[k] for k in results_df.columns if k != "Id"}
                    for _, row in results_df.iterrows()
                }

            results_df = analyze_responses(dataset, responses_by_datapointid, results_by_datapointid, semantic_equality_checker)
            results_df.to_csv(results_file, index=False)

            semantic_equality_checker.save(math_equivalence_check_path)

In [ ]:
plot_results("eedi_data", eedi_dataset, filter_unsolvable=True, filter_cutoff=False)

### With vs Without Literature-Informed Process

In [ ]:
import scipy.stats as st

# -> Table 24 (Appendix C.3): 429-solvable-problem breadth check across 9 reasoning
# models w/ and w/o the literature-informed (LS) prompt.
data_folder = "eedi_data"
results_folder = os.path.join(data_folder, "joint_results")

model_configs = [
    # (dataset_label, family_label, size_label, naive_file, enforce_file, group)
    ("Eedi", "deepseek v3.2", "685b",
     "deepseek-naive-deepseek-reasoner",
     "deepseek-enforce-process-deepseek-reasoner", 0),
    ("Eedi", "gpt-oss", "20b",
     "openrouter-naive-openai_gpt-oss-20b-reasoner",
     "openrouter-enforce-process-openai_gpt-oss-20b-reasoner", 1),
    ("Eedi", "gpt-oss", "120b",
     "openrouter-naive-openai_gpt-oss-120b-reasoner",
     "openrouter-enforce-process-openai_gpt-oss-120b-reasoner", 1),
    ("Eedi", "gpt-5", "--",
     "openai-naive-gpt-5",
     "openai-enforce-process-gpt-5", 1),
    ("Eedi", "glm 4.7 flash", "31b",
     "openrouter-naive-z-ai_glm-4.7-flash-reasoner",
     "openrouter-enforce-process-z-ai_glm-4.7-flash-reasoner", 2),
    ("Eedi", "glm 4.7", "358b",
     "openrouter-naive-z-ai_glm-4.7-reasoner",
     "openrouter-enforce-process-z-ai_glm-4.7-reasoner", 2),
    ("Eedi", "gemma 4", "31b",
     "vllm-naive-google_gemma-4-31b-it",
     "vllm-enforce-process-google_gemma-4-31b-it", 3),
    ("Eedi", "gemini 2.5 flash", "--",
     "gemini-naive-gemini-2.5-flash",
     "gemini-enforce-process-gemini-2.5-flash", 3),
    ("Eedi", "gemini 2.5 pro", "--",
     "gemini-naive-gemini-2.5-pro",
     "gemini-enforce-process-gemini-2.5-pro", 3),
]

def load_and_filter(filename):
    path = os.path.join(results_folder, f"{filename}_results.csv")
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    df = df[df["Id"].apply(lambda x: eedi_dataset[int(x)]["Problem"]["Solvable"])]
    return df

def fmt_score(df):
    if df is None:
        return r"$0.xx \pm 0.xx$", None, None
    mean, ci = get_mean_and_ci(df["proportional_match"])
    return f"${mean:.2f} \\pm {ci:.2f}$", df, mean

def reasoning_len_cells(df, is_last):
    """Return the LaTeX for the two-column reasoning-length cell pair.

    When the underlying values are all zero (model doesn't expose reasoning
    traces) we collapse the pair into a single '--' via \\multicolumn.
    """
    col_spec = "c" if is_last else "c|"
    unavailable = f"\\multicolumn{{2}}{{{col_spec}}}{{--}}"

    if df is None or "reasoning_len_chars" not in df.columns:
        return unavailable
    col = pd.to_numeric(df["reasoning_len_chars"], errors="coerce").dropna()
    if len(col) == 0:
        return unavailable
    mean, ci = get_mean_and_ci(col)
    if mean == 0 and ci == 0:
        return unavailable
    return f"${mean:.0f}$ & ${ci:.0f}$"

def make_bold(math_str):
    # Converts "$x \pm y$" -> "$\mathbf{x \pm y}$"
    inner = math_str[1:-1]  # strip surrounding $
    return f"$\\mathbf{{{inner}}}$"

rows = []
for dataset_label, family, size, naive_file, enforce_file, group in model_configs:
    naive_df = load_and_filter(naive_file)
    enforce_df = load_and_filter(enforce_file)

    naive_str, naive_df_f, naive_mean = fmt_score(naive_df)
    enforce_str, enforce_df_f, enforce_mean = fmt_score(enforce_df)

    naive_len_cells   = reasoning_len_cells(naive_df_f,   is_last=False)
    enforce_len_cells = reasoning_len_cells(enforce_df_f, is_last=True)

    naive_wins = enforce_wins = False
    if naive_df_f is not None and enforce_df_f is not None:
        common_ids = set(naive_df_f["Id"].astype(str)) & set(enforce_df_f["Id"].astype(str))
        naive_aligned = (naive_df_f[naive_df_f["Id"].astype(str).isin(common_ids)]
                         .sort_values("Id")["proportional_match"].values)
        enforce_aligned = (enforce_df_f[enforce_df_f["Id"].astype(str).isin(common_ids)]
                           .sort_values("Id")["proportional_match"].values)
        _, pvalue = st.ttest_rel(naive_aligned, enforce_aligned)
        if pvalue < 0.05:
            if naive_mean > enforce_mean:
                naive_wins = True
            else:
                enforce_wins = True

    rows.append((dataset_label, family, size,
                 naive_str, enforce_str,
                 naive_len_cells, enforce_len_cells,
                 naive_wins, enforce_wins, group))

lines = [
    r"\begin{table*}[h!]",
    r"    \centering",
    r"    \begin{tabular}{lll|cc|r@{\,$\pm$\,}l|r@{\,$\pm$\,}l}",
    r"        \toprule",
    r"         &  &  & \multicolumn{2}{c|}{\textbf{Proportional Match}} & \multicolumn{4}{c}{\textbf{Reasoning Length (chars)}} \\",
    r"        \textbf{Dataset} & \textbf{Model Family} & \textbf{Model Size} & \textbf{w/o LS} & \textbf{w/ LS} & \multicolumn{2}{c|}{\textbf{w/o LS}} & \multicolumn{2}{c}{\textbf{w/ LS}} \\",
    r"        \midrule",
]

prev_group = None
for (dataset_label, family, size, naive_str, enforce_str,
     naive_len_cells, enforce_len_cells,
     naive_wins, enforce_wins, group) in rows:
    if prev_group is not None and group != prev_group:
        lines.append(r"        \midrule")
    naive_fmt   = make_bold(naive_str)   if naive_wins   else naive_str
    enforce_fmt = make_bold(enforce_str) if enforce_wins else enforce_str
    lines.append(
        f"        {dataset_label} & {family} & {size} & {naive_fmt} & {enforce_fmt} "
        f"& {naive_len_cells} & {enforce_len_cells} \\\\"
    )
    prev_group = group

lines += [
    r"        \bottomrule",
    r"    \end{tabular}",
    r"    \caption{Proportional match scores and average reasoning length (characters) both with and without literature informed reasoning structure (LS) for different sizes of models. Reasoning enabled. Statistically significant results (two sided t-test) are marked in bold.}",
    r"    \label{tab:scores_with_without_ls_informed}",
    r"\end{table*}",
]

print("\n".join(lines))

## SciQ

In [ ]:
sciq_equivalence_check_path = "cache/science_semantic_equivalence_checker.pkl"

sciq_dataset = get_or_create_dataset("sciq_data", n_limit=500)
print(f"We have {len(sciq_dataset)} SciQ questions")

if os.path.exists(sciq_equivalence_check_path):
    print("Loading existing science semantic equality checker")
    sciq_semantic_equality_checker = ScienceSemanticEqualityChecker.load(equality_client, sciq_equivalence_check_path)
else:
    sciq_semantic_equality_checker = ScienceSemanticEqualityChecker(equality_client, equality_model_config)
    sciq_semantic_equality_checker.save(sciq_equivalence_check_path)

### Evaluation

In [ ]:
sciq_results_folder = os.path.join("sciq_data", "joint_results")

for filename in os.listdir(sciq_results_folder):
    if not filename.endswith("_responses_by_datapointid.json"): continue

    print(f"Processing responses: {os.path.join(sciq_results_folder, filename)}")
    with open(os.path.join(sciq_results_folder, filename), "r+") as f:
        responses_by_datapointid = json.loads(f.read())

        results_file = os.path.join(sciq_results_folder, f"{filename.replace('_responses_by_datapointid.json', '')}_results.csv")

        results_by_datapointid = {}
        if os.path.exists(results_file):
            results_df = pd.read_csv(results_file)
            results_by_datapointid = {
                str(row["Id"]): {k: row[k] for k in results_df.columns if k != "Id"}
                for _, row in results_df.iterrows()
            }

        results_df = analyze_responses(sciq_dataset, responses_by_datapointid, results_by_datapointid, sciq_semantic_equality_checker)
        results_df.to_csv(results_file, index=False)

        sciq_semantic_equality_checker.save(sciq_equivalence_check_path)

In [ ]:
plot_results("sciq_data", sciq_dataset, filter_unsolvable=True, filter_cutoff=False)

## Effect of revealing the correct answer

Paired comparison of `naive` vs `naive-correct` settings for DeepSeek on both Eedi and SciQ.
Reports baseline mean, reveal mean, absolute and relative gain, and a paired t-test on per-problem proportional match (Eedi restricted to solvable problems, matching the rest of the paper).

In [ ]:
import scipy.stats as st

def reveal_effect(data_folder, dataset, dataset_label, filter_solvable=True):
    base_path = os.path.join(data_folder, "joint_results", "deepseek-naive-deepseek-reasoner_results.csv")
    reveal_path = os.path.join(data_folder, "joint_results", "deepseek-naive-correct-deepseek-reasoner_results.csv")

    base = pd.read_csv(base_path)[["Id", "proportional_match"]].rename(columns={"proportional_match": "pm_base"})
    reveal = pd.read_csv(reveal_path)[["Id", "proportional_match"]].rename(columns={"proportional_match": "pm_reveal"})

    if filter_solvable:
        solvable_ids = {dpid for dpid in range(len(dataset)) if dataset[dpid]["Problem"]["Solvable"]}
        base = base[base["Id"].isin(solvable_ids)]
        reveal = reveal[reveal["Id"].isin(solvable_ids)]

    merged = base.merge(reveal, on="Id").dropna()

    mb, mr = merged.pm_base.mean(), merged.pm_reveal.mean()
    abs_gain = mr - mb
    rel_gain = abs_gain / mb * 100
    t, p = st.ttest_rel(merged.pm_reveal, merged.pm_base)

    print(f"=== {dataset_label} ===")
    print(f"N paired:      {len(merged)}")
    print(f"mean baseline: {mb:.4f}")
    print(f"mean reveal:   {mr:.4f}")
    print(f"abs gain:      {abs_gain:+.4f}")
    print(f"rel gain:      {rel_gain:+.1f}%")
    print(f"paired t-test: t={t:.3f}, p={p:.6f}")
    print()

reveal_effect("eedi_data", eedi_dataset, "Eedi")
reveal_effect("sciq_data", sciq_dataset, "SciQ", filter_solvable=False)
